In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

import joblib

In [2]:
df = pd.read_csv("../data/raw/pharmacy_sales_dataset.csv")

df.head()

,transaction_id,item,quantity,price,date,customer_id
0,1000,Ibuprofen,1,18,2023-01-01,190
1,1001,Azithromycin,2,40,2023-01-01,58
2,1002,Cetirizine,1,15,2023-01-01,174
3,1003,Atorvastatin,1,30,2023-01-01,152
4,1004,Pantoprazole,1,28,2023-01-01,8


## Data Cleaning

In [3]:
df = df.dropna()

df['date'] = pd.to_datetime(df['date'])

df.head()

,transaction_id,item,quantity,price,date,customer_id
0,1000,Ibuprofen,1,18,2023-01-01,190
1,1001,Azithromycin,2,40,2023-01-01,58
2,1002,Cetirizine,1,15,2023-01-01,174
3,1003,Atorvastatin,1,30,2023-01-01,152
4,1004,Pantoprazole,1,28,2023-01-01,8


## Feature Engineering

In [4]:
df["date"] = pd.to_datetime(df["date"])

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14590 entries, 0 to 14589
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   transaction_id  14590 non-null  int64         
 1   item            14590 non-null  object        
 2   quantity        14590 non-null  int64         
 3   price           14590 non-null  int64         
 4   date            14590 non-null  datetime64[ns]
 5   customer_id     14590 non-null  int64         
dtypes: datetime64[ns](1), int64(4), object(1)
memory usage: 684.0+ KB


In [5]:
df["total_price"] = df["quantity"] * df["price"]

df.head()

,transaction_id,item,quantity,price,date,customer_id,total_price
0,1000,Ibuprofen,1,18,2023-01-01,190,18
1,1001,Azithromycin,2,40,2023-01-01,58,80
2,1002,Cetirizine,1,15,2023-01-01,174,15
3,1003,Atorvastatin,1,30,2023-01-01,152,30
4,1004,Pantoprazole,1,28,2023-01-01,8,28


In [6]:
daily_sales = df.groupby("date").agg({
    "total_price":"sum",
    "quantity":"sum",
    "transaction_id":"count"
}).reset_index()

In [7]:
daily_sales.columns = ["date","total_sales","total_quantity","num_transactions"]

In [8]:
daily_sales["day"] = daily_sales["date"].dt.day
daily_sales["month"] = daily_sales["date"].dt.month
daily_sales["day_of_week"] = daily_sales["date"].dt.dayofweek

daily_sales.head()

,date,total_sales,total_quantity,num_transactions,day,month,day_of_week
0,2023-01-01,4293,172,60,1,1,6
1,2023-01-02,2580,102,35,2,1,0
2,2023-01-03,3470,138,44,3,1,1
3,2023-01-04,2084,84,29,4,1,2
4,2023-01-05,2249,79,27,5,1,3


In [9]:
X = daily_sales[[
    "day",
    "month",
    "day_of_week",
    "total_quantity",
    "num_transactions"
]]
y = daily_sales["total_sales"]

## Train/Test Split

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Train Regression Model

In [11]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(X_train, y_train)

LinearRegression()

## Evaluate Model

In [12]:
pred = model.predict(X_test)

print("R2 Score:", r2_score(y_test, pred))

print("RMSE:", np.sqrt(mean_squared_error(y_test, pred)))

R2 Score: 0.9573355175486874
RMSE: 187.96075753072893


## Predict Future Sales

## Save Model

In [14]:
joblib.dump(model,"../models/sales_model.pkl")

['../models/sales_model.pkl']